# Conditional disappearance: does one seizure type change whether another disappears?

Runs on one **unit** (`UNIT`, the first code cell; see `cc_stats/units.py`): for `types` the five core groups, for `codes` the 7 codes with at least 20 patients before surgery (focal awareness codes aside). The types notebook is the source; the codes one is generated from it by `analysis/unit_notebooks.py` (only `UNIT` and the notes differ), so edit the types notebook and regenerate. Below, "type" means a group of the unit.

For each **target** type B and **condition** A (another core type, or LGS): among patients who had B before surgery and whose post-op types were recorded, the share in whom B disappeared, **with A vs without A** before surgery.

- Targets and conditions: the unit's groups (for types the five core groups: atonic, absence, BTC, tonic, myoclonic), plus LGS as a condition. The core groups share no seizure codes and the codes are distinct, so no pair is circular.
- Each pair: Wilson 95% CIs and Fisher's exact test. **Holm-corrected** across every pair, since this many comparisons will throw up some striking differences by chance.
- `cooccurrence.ipynb` (in this folder) shows which types occur together; this notebook asks what that co-occurrence does to disappearance. `combinations_and_outcome.ipynb` asks the same of improvement.

`conditional_disappearance.py` computes the table, and this notebook runs it. The output goes to `results/<version>/4_combinations/<unit>/conditional_disappearance.csv`.

In [ ]:
# The unit this notebook runs on: 'types' or 'codes' (cc_stats/units.py).
# Set here only; the codes notebook is generated with 'codes' (analysis/unit_notebooks.py).
UNIT = 'codes'

In [ ]:
import subprocess
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from cc_stats.paths import DATASET, PROJECT_ROOT, results_dir
from cc_stats.paths import ANALYSIS_CSV
from cc_stats.plots import PAIR_A, PAIR_B, pair_heatmap, paired_rate_plot, pretty, save_figure
from cc_stats.units import unit as get_unit

print(f'dataset {DATASET}')
run = subprocess.run([sys.executable, 'analysis/4_combinations/conditional_disappearance.py', UNIT],
                     capture_output=True, text=True, cwd=PROJECT_ROOT)
if run.returncode != 0:
    raise RuntimeError(f'conditional_disappearance.py failed:\n{run.stderr}')
unit = get_unit(UNIT, pd.read_csv(ANALYSIS_CSV, index_col='subject_id'))
# 'type' or 'code', for figure titles and labels
WORD = 'type' if UNIT == 'types' else 'code'
FOLDER = f'4_combinations/{UNIT}'
pairs = pd.read_csv(results_dir(FOLDER) / 'conditional_disappearance.csv')
TARGETS = unit.pair_groups
CONDITIONS = TARGETS + ['lgs']
# Heatmaps grow with the number of rows (drawn for 5 targets)
big = max(1, len(TARGETS) / 5)
# The reference deck's pair, BTC disappearing by atonic; for codes, the specific codes it used:
# generalized tonic-clonic (2) by generalized atonic (8)
REFERENCE = ('btc', 'atonic') if UNIT == 'types' else ('code_02', 'code_08')


def fmt_p(v):
    return '<0.001' if v < 0.001 else f'{v:.3f}'


def pct(k, n):
    return f'{k}/{n} ({k / n:.0%})' if n else '-'

## Every pair

Each cell: disappearance of the row's type **with** the column's condition minus **without** it, in percentage points. Red means the target disappears more often when the condition is present. Cells with fewer than 5 patients on either side are greyed out.

In [ ]:
ax = pair_heatmap(pairs, TARGETS, CONDITIONS, figsize=(10 * big, 6.5 * big))
ax.set_title('Disappearance with vs without each condition\n(* uncorrected p < 0.05; none survive Holm)'
             if (pairs['p_holm'] >= 0.05).all() else
             'Disappearance with vs without each condition (* uncorrected p < 0.05)')
plt.tight_layout()
save_figure(FOLDER, 'conditional_disappearance_heatmap')
plt.show()

## Strongest pairs

Sorted by uncorrected p. The Holm column is the one to trust.

In [ ]:
top = pairs.sort_values('p').head(8)
display(pd.DataFrame({
    'Target disappears': top['target'],
    'Condition': top['condition'],
    'With condition': [pct(k, n) for k, n in top[['left_with', 'n_with']].to_numpy()],
    'Without': [pct(k, n) for k, n in top[['left_without', 'n_without']].to_numpy()],
    'Difference': (top['difference'] * 100).round().astype(int).astype(str) + ' pts',
    'p': top['p'].map(fmt_p),
    'Holm p': top['p_holm'].map(fmt_p),
}).style.hide(axis='index'))

## Two pairs side by side

- The strongest pair in this cohort, other than the reference pair (for codes, the reference pair is itself the strongest).
- **BTC by atonic**: the pair in the reference deck (generalized tonic-clonic by atonic, 70% vs 24% on the v1 data, specific codes). For codes, it is that pair of codes: generalized tonic-clonic (2) by generalized atonic (8).

In [ ]:
is_reference = (pairs['target'] == REFERENCE[0]) & (pairs['condition'] == REFERENCE[1])
reference = pairs[is_reference].iloc[0]
strongest = pairs[~is_reference].sort_values('p').iloc[0]
# paired_rate_plot draws the first row at the bottom, so the reference pair goes first
rows = pd.DataFrame([reference, strongest])
labels = rows['target'] + ' disappears, by ' + rows['condition']

fig, ax = plt.subplots(figsize=(11, 3.4))
paired_rate_plot(ax, labels.tolist(), top=('with condition', rows['left_with'], rows['n_with'], PAIR_A),
                 bottom=('without', rows['left_without'], rows['n_without'], PAIR_B), header='Disappeared',
                 p=rows['p'], p_holm=rows['p_holm'])
ax.set_xlabel(f'Patients whose seizure {WORD} disappeared (Wilson 95% CI)')
ax.set_title(f"{pretty(REFERENCE[0])} by {pretty(REFERENCE[1])} (the reference deck's example),\n"
             'and the strongest other pair', loc='left', pad=28)
ax.legend(loc='lower left', bbox_to_anchor=(0, 1.0), ncol=2, frameon=False, handletextpad=0.3)
plt.tight_layout()
save_figure(FOLDER, 'strongest_pair_vs_btc_by_atonic')
plt.show()

## Reading the results

Checked 2026-10-21 on v2, over the 7 codes with at least 20 patients as targets and conditions (plus LGS): 49 pairs.

- **Nothing survives Holm.** Each comparison is between a few dozen patients, often with fewer than 10 on one side.
- **The reference deck's pair is the strongest:** generalized tonic-clonic (2) disappeared in 10 of 14 patients who also had generalized atonic seizures (71%) vs 5 of 25 without (20%), p = 0.002, Holm p = 0.12. That is close to the deck's 70% vs 24% on the v1 data. By type, BTC by atonic is much weaker (7/22 vs 6/39, p = 0.19): the BTC type adds focal to BTC and myoclonic-tonic-clonic, and the atonic type adds focal atonic, which dilute the code-level pair.
- **The strongest other pair rests on 2 patients** (focal motor to BTC disappearing in both patients with generalized myoclonic, vs 1 of 16 without); read it as noise.
- Generalized spasms (9) also disappeared more often with generalized atonic (3/5 vs 1/16, p = 0.028).
- A difference here is a lead for the next study, not a finding: it says which combinations to look at with more patients (or with EEG), not that one seizure type causes another to resolve.